
# Register Roboflow model via MLflow

**Author:** Juliana Thomasia C. Marcos\
**Input:** Roboflow project, specific model information and configuration parameters\
**Output:** Registered MLflow model in Databricks Unity Catalog, associated model assets, and sample predictions

## Overview

This notebook retrieves a trained model from Roboflow and registers it in the Databricks MLflow Model Registry for subsequent inference workflows. It downloads the model weights and sample images from Roboflow, stores the associated assets in a designated Databricks volume, and organizes registered models based on crop, trait, and model type.

Before registration, the notebook checks whether the specified Roboflow model version is already registered to avoid duplicate entries. If the model is new, it is registered as a new version and assigned the `champion` alias; otherwise, the existing version is reused and assigned the alias. For newly registered models, sample inference is performed to validate the registration and generate prediction outputs for inspection.


# Packages import and installation

In [0]:
%pip install ..

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
import os
import requests
import zipfile
import subprocess
import time
import pandas as pd
import shutil

from mlflow import MlflowClient

from ag_vision.constants import paths
from ag_vision.data_io.roboflow_io import get_export_link, download_zip
from ag_vision.data_io.mlflow_io import roboflow_to_coco, write_coco_output_to_catalog, annotate_images, existing_version_with_tag, register_roboflow_model

# Variables set up

In [0]:
client = MlflowClient()
mlflow.set_registry_uri("databricks-uc")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
MODELS_SCHEMA = "cv_models_registry"
CV_MODELS_SCHEMA_NAME = f"{catalog_name}.{MODELS_SCHEMA}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CV_MODELS_SCHEMA_NAME}")

In [0]:
secret_scope = "enablement-package"
parameters = dbutils.widgets.getAll()

In [0]:
roboflow_model_id = parameters['model_id']
workspace_name = "cgiar-workspace"
roboflow_api_key = dbutils.secrets.get(scope = secret_scope, key = "roboflow_api_key")
roboflow_model_name = f"{parameters['project_name']}__v{parameters['version']}"

In [0]:
alias = "champion"
model_type = parameters['model_type']
reg_model_name = f"{parameters['trait']}__{parameters['crop_type']}__{parameters['model_type']}"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CV_MODELS_SCHEMA_NAME}.{reg_model_name}")

In [0]:
reg_model_path = f"/Volumes/{catalog_name}/{MODELS_SCHEMA}/{reg_model_name}/{roboflow_model_name}"
samples_path = f"{reg_model_path}/dataset"
if not os.path.exists(samples_path):
    os.makedirs(samples_path)

# Download dataset samples from Roboflow

In [0]:
link = get_export_link(workspace_name,
                       parameters["project_name"],
                       parameters["version"],
                       parameters["model_type"],
                       roboflow_api_key)

zp = download_zip(link, f"{reg_model_path}/samples.zip")

In [0]:
parameters['model_id']

In [0]:
zf = zipfile.ZipFile(f"{reg_model_path}/samples.zip")
train_members = [m for m in zf.namelist() if m.startswith("test/")]
zf.extractall(f"{reg_model_path}/dataset", members=train_members[:10])

In [0]:
image_samples_path = f"{reg_model_path}/dataset/test/images"
image_samples = os.listdir(image_samples_path)

# Get model and weights from Roboflow

In [0]:
from inference import get_model

In [0]:
model = get_model(model_id=roboflow_model_id, api_key=roboflow_api_key)  # triggers download into the volume

In [0]:
model_weights_path = f"{reg_model_path}/weights"
if not os.path.exists(model_weights_path):
    os.makedirs(model_weights_path)
shutil.copytree(os.environ["INFERENCE_HOME"], model_weights_path, dirs_exist_ok=True)

In [0]:
full_model_weights_path = f"{model_weights_path}"
paths.show_volume_folder_link(path=full_model_weights_path,
                                url_name="here",
                                prefix=f"The Roboflow model's assests should now be available ")

# Verify availability on Databricks of the model specified

In [0]:
reg_full_model_name = f"{CV_MODELS_SCHEMA_NAME}.{reg_model_name}"

In [0]:
existing = existing_version_with_tag(reg_full_model_name, roboflow_model_id)
if existing:
    print("A model with this tag already exists.")
else:
    print(f"No model found for the combination of {parameters['crop_type']} crop and {parameters['trait']} trait, in particular for the {alias} tag. The model specified so far will be registered as a new version.")

# Register new Roboflow model as champion

In [0]:
predictions_path = None

if existing:
    print(f"Already registered as {reg_full_model_name} v{existing.version}; skipping.")
    version = existing.version
    client.set_registered_model_alias(reg_full_model_name, alias, version)
else:
    predictions_path = register_roboflow_model(reg_model_name=reg_model_name,
                                               reg_full_model_name=reg_full_model_name,
                                               roboflow_model_id=roboflow_model_id,
                                               model_weights_path=model_weights_path,
                                               image_samples_path=image_samples_path,
                                               image_samples=image_samples,
                                               model=model,
                                               parameters=parameters,
                                               client=client,
                                               roboflow_to_coco=roboflow_to_coco,
                                               write_coco_output_to_catalog=write_coco_output_to_catalog,
                                               alias=alias,
    )

In [0]:
if predictions_path:
    paths.show_volume_folder_link(path=model_weights_path,
                                    url_name="here",
                                    prefix=f"The model was registered successfully in ML flow and should be available along its assets path i.e., "
                                    )


    annotate_images(image_samples_path,
                    f"{predictions_path}",
                    parameters["model_type"],
                    parameters["model_id"]
                    )

    paths.show_volume_folder_link(path=predictions_path,
                                    url_name="here",
                                    prefix=f"The predictions for the test carried out to confirm successful model registration should now be available "
                                    )
else:
    print("Something probably went wrong during the model registration.")
